# Chapitre 3 - Exercices 3.6 à 3.10

Interprétation bayésienne de ridge (3.6, 3.7), lien entre décomposition QR et SVD (3.8),
et mises en œuvre efficaces de la sélection pas à pas ascendante et descendante (3.9, 3.10).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
set_style()

import time
import pandas as pd
from scipy import stats
from esl import ch03_regression as cr

Xtr, ytr, Xte, yte, names = cr.load_prostate_standardized()

## Exercice 3.6 : Ridge comme moyenne a posteriori

**Énoncé.** Montrer que l'estimateur ridge est la moyenne (et le mode) de la loi a
posteriori de $\beta$ sous la loi a priori gaussienne $\beta \sim \mathcal{N}(0, \tau\mathbf{I})$
et le modèle d'échantillonnage gaussien $\mathbf{y} \sim \mathcal{N}(\mathbf{X}\beta, \sigma^2\mathbf{I})$.
Trouver la relation entre le paramètre $\lambda$ de ridge et les variances $\tau$ et $\sigma^2$.

**Rappels.**
- *Formule de Bayes* : la densité a posteriori est proportionnelle au produit
  vraisemblance × densité a priori : $\pi(\beta\mid\mathbf{y}) \propto p(\mathbf{y}\mid\beta)\,\pi(\beta)$
  (la constante ne dépend pas de $\beta$).
- Une densité de la forme $\propto \exp\bigl(-\tfrac12(\beta - m)^T\mathbf{P}(\beta - m)\bigr)$,
  $\mathbf{P}$ définie positive, est celle de $\mathcal{N}(m, \mathbf{P}^{-1})$ ; sa moyenne et son
  mode valent $m$.

On suppose (comme pour ridge) les entrées centrées et l'on omet la constante ; ici $\tau$
désigne une **variance**.

**Calcul.** Par Bayes,

$$
\pi(\beta\mid\mathbf{y}) \propto \exp\Bigl(-\frac{\lVert\mathbf{y} - \mathbf{X}\beta\rVert^2}{2\sigma^2}\Bigr)\exp\Bigl(-\frac{\lVert\beta\rVert^2}{2\tau}\Bigr)
 = \exp\Bigl(-\frac{1}{2\sigma^2}\Bigl[\lVert\mathbf{y} - \mathbf{X}\beta\rVert^2 + \frac{\sigma^2}{\tau}\lVert\beta\rVert^2\Bigr]\Bigr).
$$

Posons $\lambda = \sigma^2/\tau$ et développons le crochet :

$$
\beta^T(\mathbf{X}^T\mathbf{X} + \lambda\mathbf{I})\beta - 2\beta^T\mathbf{X}^T\mathbf{y} + \mathbf{y}^T\mathbf{y}.
$$

On **complète le carré** : avec $\mathbf{A} = \mathbf{X}^T\mathbf{X} + \lambda\mathbf{I}$ (définie
positive) et $m = \mathbf{A}^{-1}\mathbf{X}^T\mathbf{y}$,

$$
\beta^T\mathbf{A}\beta - 2\beta^T\mathbf{A}m = (\beta - m)^T\mathbf{A}(\beta - m) - m^T\mathbf{A}m .
$$

Le terme $m^T\mathbf{A}m$ et $\mathbf{y}^T\mathbf{y}$ ne dépendent pas de $\beta$ et partent dans
la constante. Donc

$$
\beta\mid\mathbf{y} \sim \mathcal{N}\bigl(m,\ \sigma^2\mathbf{A}^{-1}\bigr),\qquad
m = (\mathbf{X}^T\mathbf{X} + \lambda\mathbf{I})^{-1}\mathbf{X}^T\mathbf{y} = \hat\beta^{\mathrm{ridge}}
\quad\text{avec}\quad \boxed{\lambda = \sigma^2/\tau}.
$$

La loi a posteriori étant gaussienne, sa **moyenne** et son **mode** coïncident : tous deux
valent $\hat\beta^{\mathrm{ridge}}$. $\square$

> **Intuition.** Un a priori très diffus ($\tau \to \infty$) donne $\lambda \to 0$ : les
> moindres carrés. Un a priori resserré autour de 0 ($\tau$ petit) ou un bruit fort
> ($\sigma^2$ grand) poussent vers un fort rétrécissement.

**Vérification par Monte-Carlo.** Avec $p = 2$, on tire $\beta^{(b)}$ selon l'a priori et on
les pondère par la vraisemblance (échantillonnage d'importance) : la moyenne pondérée
approche la moyenne a posteriori.

In [2]:
rng = np.random.RandomState(0)
N, p, sigma, tau = 15, 2, 1.5, 0.5
X = rng.randn(N, p) @ np.array([[1.0, 0.7], [0.0, 0.7]])
X -= X.mean(0)
beta_true = rng.randn(p) * np.sqrt(tau)
y = X @ beta_true + sigma * rng.randn(N)
lam = sigma**2 / tau
beta_ridge = np.linalg.solve(X.T @ X + lam * np.eye(p), X.T @ y)
draws = rng.randn(400000, p) * np.sqrt(tau)
loglik = -np.sum((y[None, :] - draws @ X.T) ** 2, axis=1) / (2 * sigma**2)
w = np.exp(loglik - loglik.max()); w /= w.sum()
post_mean = w @ draws
print("lambda = sigma^2/tau = %.2f" % lam)
print("moyenne a posteriori (Monte-Carlo) :", post_mean.round(3))
print("estimateur ridge                   :", beta_ridge.round(3))

lambda = sigma^2/tau = 4.50
moyenne a posteriori (Monte-Carlo) : [ 0.292 -0.104]
estimateur ridge                   : [ 0.292 -0.105]
